In [ ]:
!nvidia-smi
!pip install -q transformers accelerate bitsandbytes

Tue Sep 29 00:42:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

model_name = "Qwen/Qwen2.5-3B-Instruct"

print("Loading:", model_name)
start = time.time()

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

load_time = time.time() - start

print(f"Model loaded successfully in {load_time:.1f} seconds.")
print("Device:", model.device)
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Loading: Qwen/Qwen2.5-3B-Instruct


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully in 96.7 seconds.
Device: cuda:0
GPU memory allocated: 5.75 GB


In [ ]:
import time
import torch

prompt = """You are given a scientific claim.

Claim: ALDH1 expression is associated with poorer prognosis in breast cancer.

In one short sentence, state whether the claim is supported and briefly explain why."""

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to("cuda")

torch.cuda.synchronize()
start = time.time()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

torch.cuda.synchronize()
generation_time = time.time() - start

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

tokens_generated = len(new_tokens)
tokens_per_second = tokens_generated / generation_time

print("ANSWER:")
print(answer)

print("\n--- PERFORMANCE ---")
print(f"Generation time: {generation_time:.2f} seconds")
print(f"Tokens generated: {tokens_generated}")
print(f"Speed: {tokens_per_second:.2f} tokens/second")

ANSWER:
The claim is supported because there is substantial evidence from various studies showing that higher levels of ALDH1 (Aldehyde Dehydrogenase 1) expression are often correlated with more aggressive breast cancer behavior and poorer patient outcomes.

--- PERFORMANCE ---
Generation time: 7.60 seconds
Tokens generated: 46
Speed: 6.05 tokens/second


In [ ]:
import torch
import gc
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Remove the 3B model from GPU memory
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

model_name = "Qwen/Qwen2.5-7B-Instruct"

# 4-bit loading so the 7B model fits comfortably on the T4
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading:", model_name)
start = time.time()

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

load_time = time.time() - start

print(f"Model loaded successfully in {load_time:.1f} seconds.")
print("Device:", model.device)
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Loading: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully in 356.9 seconds.
Device: cuda:0
GPU memory allocated: 5.46 GB


In [ ]:
!wget -q https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz
!tar -xzf data.tar.gz

print("SciFact downloaded and extracted.")
!ls data


SciFact downloaded and extracted.
claims_dev.jsonl   claims_train.jsonl  cross_validation
claims_test.jsonl  corpus.jsonl


In [ ]:
import json

# Load SciFact corpus
corpus = {}
with open("data/corpus.jsonl", "r") as f:
    for line in f:
        doc = json.loads(line)
        corpus[doc["doc_id"]] = doc

# Load development claims
claims = []
with open("data/claims_dev.jsonl", "r") as f:
    for line in f:
        claims.append(json.loads(line))

print(f"Loaded {len(corpus)} scientific abstracts.")
print(f"Loaded {len(claims)} development claims.")

print("\nExample claim:")
print(claims[0])

Loaded 5183 scientific abstracts.
Loaded 300 development claims.

Example claim:
{'id': 1, 'claim': '0-dimensional biomaterials show inductive properties.', 'evidence': {}, 'cited_doc_ids': [31715818]}


In [ ]:
claims_with_evidence = [
    claim for claim in claims
    if claim.get("evidence")
]

print(f"Total dev claims: {len(claims)}")
print(f"Claims with gold evidence: {len(claims_with_evidence)}")

print("\nExample claim with evidence:")
print(claims_with_evidence[0])

Total dev claims: 300
Claims with gold evidence: 188

Example claim with evidence:
{'id': 3, 'claim': '1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.', 'evidence': {'14717500': [{'sentences': [2, 5], 'label': 'SUPPORT'}, {'sentences': [7], 'label': 'SUPPORT'}]}, 'cited_doc_ids': [14717500]}


In [ ]:
import time
import torch

# Take one real SciFact claim with annotated evidence
example = claims_with_evidence[0]

claim_text = example["claim"]

# Get the first gold evidence document ID
gold_doc_id = int(next(iter(example["evidence"])))

# Get the actual scientific abstract from SciFact
doc = corpus[gold_doc_id]
abstract_text = " ".join(doc["abstract"])

# Get the actual gold label
gold_label = example["evidence"][str(gold_doc_id)][0]["label"]

prompt = f"""You are verifying a scientific claim using the provided evidence.

Claim:
{claim_text}

Evidence:
{abstract_text}

Classify the claim as exactly one of:
SUPPORT
CONTRADICT

Return only the label."""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to("cuda")

torch.cuda.synchronize()
start = time.time()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

torch.cuda.synchronize()
generation_time = time.time() - start

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
prediction = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("CLAIM:")
print(claim_text)

print("\nGOLD LABEL:")
print(gold_label)

print("\nQWEN 7B PREDICTION:")
print(prediction)

print(f"\nGeneration time: {generation_time:.2f} seconds")

CLAIM:
1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.

GOLD LABEL:
SUPPORT

QWEN 7B PREDICTION:
CONTRADICT

Generation time: 1.52 seconds


In [ ]:
import json
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# --------------------------------------------------
# LOAD OUR 20-CLAIM TRIAL DATA
# --------------------------------------------------

with open("trial_retrieval_results.json", "r", encoding="utf-8") as f:
    trial_data = json.load(f)

print(f"Loaded {len(trial_data)} trial claims.")


# --------------------------------------------------
# LOAD QWEN 2.5 3B IN 4-BIT
# --------------------------------------------------

model_name = "Qwen/Qwen2.5-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading Qwen 2.5 3B...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Qwen 3B loaded.")


# --------------------------------------------------
# FUNCTION TO ASK QWEN TO VERIFY ONE CLAIM
# --------------------------------------------------

def verify_claim(claim, documents):

    evidence_text = ""

    for i, doc in enumerate(documents, start=1):
        evidence_text += (
            f"\nDOCUMENT {i}\n"
            f"Title: {doc['title']}\n"
            f"Abstract: {doc['text']}\n"
        )

    prompt = f"""
You are verifying a scientific claim using retrieved scientific abstracts.

CLAIM:
{claim}

RETRIEVED EVIDENCE:
{evidence_text}

Based only on the retrieved evidence, classify the claim as exactly one of:

SUPPORT
CONTRADICT

Return only the label.
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    generated_tokens = outputs[
        :,
        inputs["input_ids"].shape[1]:
    ]

    prediction = tokenizer.decode(
        generated_tokens[0],
        skip_special_tokens=True
    ).strip().upper()

    if "CONTRADICT" in prediction:
        return "CONTRADICT"

    if "SUPPORT" in prediction:
        return "SUPPORT"

    return "INVALID"


# --------------------------------------------------
# RUN BOTH RETRIEVERS THROUGH QWEN 3B
# --------------------------------------------------

results = []

start_time = time.time()

for number, item in enumerate(trial_data, start=1):

    print(f"Processing claim {number}/20...")

    bm25_prediction = verify_claim(
        item["claim"],
        item["bm25_top5"]
    )

    dense_prediction = verify_claim(
        item["claim"],
        item["dense_top5"]
    )

    results.append({
        "claim_id": item["claim_id"],
        "claim": item["claim"],
        "gold_label": item["gold_label"],
        "bm25_prediction": bm25_prediction,
        "dense_prediction": dense_prediction
    })


# --------------------------------------------------
# CALCULATE ACCURACY
# --------------------------------------------------

bm25_correct = sum(
    r["bm25_prediction"] == r["gold_label"]
    for r in results
)

dense_correct = sum(
    r["dense_prediction"] == r["gold_label"]
    for r in results
)

elapsed = time.time() - start_time

print()
print("--------------------------------")
print("QWEN 3B TRIAL RESULTS")
print("--------------------------------")

print(
    f"BM25 + Qwen 3B: "
    f"{bm25_correct}/20 = {bm25_correct / 20 * 100:.2f}%"
)

print(
    f"Dense + Qwen 3B: "
    f"{dense_correct}/20 = {dense_correct / 20 * 100:.2f}%"
)

print(f"Total generation time: {elapsed:.2f} seconds")


# --------------------------------------------------
# SAVE RESULTS
# --------------------------------------------------

with open("qwen_3b_trial_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to qwen_3b_trial_results.json")

Loaded 20 trial claims.
Loading Qwen 2.5 3B...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen 3B loaded.
Processing claim 1/20...
Processing claim 2/20...
Processing claim 3/20...
Processing claim 4/20...
Processing claim 5/20...
Processing claim 6/20...
Processing claim 7/20...
Processing claim 8/20...
Processing claim 9/20...
Processing claim 10/20...
Processing claim 11/20...
Processing claim 12/20...
Processing claim 13/20...
Processing claim 14/20...
Processing claim 15/20...
Processing claim 16/20...
Processing claim 17/20...
Processing claim 18/20...
Processing claim 19/20...
Processing claim 20/20...

--------------------------------
QWEN 3B TRIAL RESULTS
--------------------------------
BM25 + Qwen 3B: 10/20 = 50.00%
Dense + Qwen 3B: 9/20 = 45.00%
Total generation time: 80.39 seconds
Saved to qwen_3b_trial_results.json


In [ ]:
import json
import time
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


# --------------------------------------------------
# 1. REMOVE QWEN 3B FROM GPU MEMORY
# --------------------------------------------------

try:
    del model
    del tokenizer
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print("Previous model cleared from GPU.")


# --------------------------------------------------
# 2. LOAD OUR SAME 20 CLAIMS
# --------------------------------------------------

with open("trial_retrieval_results.json", "r", encoding="utf-8") as f:
    trial_data = json.load(f)

print(f"Loaded {len(trial_data)} trial claims.")


# --------------------------------------------------
# 3. LOAD QWEN 2.5 7B IN 4-BIT
# --------------------------------------------------

model_name = "Qwen/Qwen2.5-7B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading Qwen 2.5 7B...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Qwen 7B loaded.")


# --------------------------------------------------
# 4. CLAIM VERIFICATION FUNCTION
# --------------------------------------------------

def verify_claim(claim, documents):

    evidence_text = ""

    for i, doc in enumerate(documents, start=1):
        evidence_text += (
            f"\nDOCUMENT {i}\n"
            f"Title: {doc['title']}\n"
            f"Abstract: {doc['text']}\n"
        )

    prompt = f"""
You are verifying a scientific claim using retrieved scientific abstracts.

CLAIM:
{claim}

RETRIEVED EVIDENCE:
{evidence_text}

Based only on the retrieved evidence, classify the claim as exactly one of:

SUPPORT
CONTRADICT

Return only the label.
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    generated_tokens = outputs[
        :,
        inputs["input_ids"].shape[1]:
    ]

    prediction = tokenizer.decode(
        generated_tokens[0],
        skip_special_tokens=True
    ).strip().upper()

    if "CONTRADICT" in prediction:
        return "CONTRADICT"

    if "SUPPORT" in prediction:
        return "SUPPORT"

    return "INVALID"


# --------------------------------------------------
# 5. RUN BOTH RETRIEVERS THROUGH QWEN 7B
# --------------------------------------------------

results = []

start_time = time.time()

for number, item in enumerate(trial_data, start=1):

    print(f"Processing claim {number}/20...")

    bm25_prediction = verify_claim(
        item["claim"],
        item["bm25_top5"]
    )

    dense_prediction = verify_claim(
        item["claim"],
        item["dense_top5"]
    )

    results.append({
        "claim_id": item["claim_id"],
        "claim": item["claim"],
        "gold_label": item["gold_label"],
        "bm25_prediction": bm25_prediction,
        "dense_prediction": dense_prediction
    })


# --------------------------------------------------
# 6. CALCULATE ACCURACY
# --------------------------------------------------

bm25_correct = sum(
    r["bm25_prediction"] == r["gold_label"]
    for r in results
)

dense_correct = sum(
    r["dense_prediction"] == r["gold_label"]
    for r in results
)

elapsed = time.time() - start_time

print()
print("--------------------------------")
print("QWEN 7B TRIAL RESULTS")
print("--------------------------------")

print(
    f"BM25 + Qwen 7B: "
    f"{bm25_correct}/20 = {bm25_correct / 20 * 100:.2f}%"
)

print(
    f"Dense + Qwen 7B: "
    f"{dense_correct}/20 = {dense_correct / 20 * 100:.2f}%"
)

print(f"Total generation time: {elapsed:.2f} seconds")


# --------------------------------------------------
# 7. SAVE RESULTS
# --------------------------------------------------

with open("qwen_7b_trial_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to qwen_7b_trial_results.json")

Previous model cleared from GPU.
Loaded 20 trial claims.
Loading Qwen 2.5 7B...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen 7B loaded.
Processing claim 1/20...
Processing claim 2/20...
Processing claim 3/20...
Processing claim 4/20...
Processing claim 5/20...
Processing claim 6/20...
Processing claim 7/20...
Processing claim 8/20...
Processing claim 9/20...
Processing claim 10/20...
Processing claim 11/20...
Processing claim 12/20...
Processing claim 13/20...
Processing claim 14/20...
Processing claim 15/20...
Processing claim 16/20...
Processing claim 17/20...
Processing claim 18/20...
Processing claim 19/20...
Processing claim 20/20...

--------------------------------
QWEN 7B TRIAL RESULTS
--------------------------------
BM25 + Qwen 7B: 16/20 = 80.00%
Dense + Qwen 7B: 15/20 = 75.00%
Total generation time: 126.55 seconds
Saved to qwen_7b_trial_results.json


In [ ]:
import json
import time
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# --------------------------------------------------
# 1. CLEAR CURRENT MODEL
# --------------------------------------------------

try:
    del model
    del tokenizer
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print("Previous model cleared from GPU.")


# --------------------------------------------------
# 2. LOAD FINAL 188-CLAIM DATA
# --------------------------------------------------

with open("final_retrieval_results.json", "r", encoding="utf-8") as f:
    final_data = json.load(f)

print(f"Loaded {len(final_data)} claims.")


# --------------------------------------------------
# 3. LOAD QWEN 2.5 3B IN 4-BIT
# --------------------------------------------------

model_name = "Qwen/Qwen2.5-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading Qwen 2.5 3B...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Qwen 3B loaded.")


# --------------------------------------------------
# 4. VERIFICATION FUNCTION
# --------------------------------------------------

def verify_claim(claim, documents):

    evidence_text = ""

    for i, doc in enumerate(documents, start=1):
        evidence_text += (
            f"\nDOCUMENT {i}\n"
            f"Title: {doc['title']}\n"
            f"Abstract: {doc['text']}\n"
        )

    prompt = f"""
You are verifying a scientific claim using retrieved scientific abstracts.

CLAIM:
{claim}

RETRIEVED EVIDENCE:
{evidence_text}

Based only on the retrieved evidence, classify the claim as exactly one of:

SUPPORT
CONTRADICT

Return only the label.
"""

    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated_tokens[0],
        skip_special_tokens=True
    ).strip().upper()

    if "CONTRADICT" in prediction:
        return "CONTRADICT"

    if "SUPPORT" in prediction:
        return "SUPPORT"

    return "INVALID"


# --------------------------------------------------
# 5. RUN ALL 188 CLAIMS
# --------------------------------------------------

results = []
start_time = time.time()

for number, item in enumerate(final_data, start=1):

    print(f"Processing claim {number}/{len(final_data)}...")

    bm25_prediction = verify_claim(
        item["claim"],
        item["bm25_top5"]
    )

    dense_prediction = verify_claim(
        item["claim"],
        item["dense_top5"]
    )

    results.append({
        "claim_id": item["claim_id"],
        "claim": item["claim"],
        "gold_label": item["gold_label"],
        "bm25_prediction": bm25_prediction,
        "dense_prediction": dense_prediction
    })


# --------------------------------------------------
# 6. RESULTS
# --------------------------------------------------

bm25_correct = sum(
    r["bm25_prediction"] == r["gold_label"]
    for r in results
)

dense_correct = sum(
    r["dense_prediction"] == r["gold_label"]
    for r in results
)

elapsed = time.time() - start_time

print()
print("--------------------------------")
print("QWEN 3B FINAL RESULTS")
print("--------------------------------")

print(
    f"BM25 + Qwen 3B: "
    f"{bm25_correct}/188 = {bm25_correct / 188 * 100:.2f}%"
)

print(
    f"Dense + Qwen 3B: "
    f"{dense_correct}/188 = {dense_correct / 188 * 100:.2f}%"
)

print(f"Total generation time: {elapsed:.2f} seconds")


# --------------------------------------------------
# 7. SAVE
# --------------------------------------------------

with open("qwen_3b_final_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to qwen_3b_final_results.json")

Exception ignored in: <function tqdm.__del__ at 0x7eb9a14739c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tqdm/std.py", line 1147, in __del__
    def __del__(self):
KeyboardInterrupt: 


Previous model cleared from GPU.
Loaded 188 claims.
Loading Qwen 2.5 3B...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen 3B loaded.
Processing claim 1/188...
Processing claim 2/188...
Processing claim 3/188...
Processing claim 4/188...
Processing claim 5/188...
Processing claim 6/188...
Processing claim 7/188...
Processing claim 8/188...
Processing claim 9/188...
Processing claim 10/188...
Processing claim 11/188...
Processing claim 12/188...
Processing claim 13/188...
Processing claim 14/188...
Processing claim 15/188...
Processing claim 16/188...
Processing claim 17/188...
Processing claim 18/188...
Processing claim 19/188...
Processing claim 20/188...
Processing claim 21/188...
Processing claim 22/188...
Processing claim 23/188...
Processing claim 24/188...
Processing claim 25/188...
Processing claim 26/188...
Processing claim 27/188...
Processing claim 28/188...
Processing claim 29/188...
Processing claim 30/188...
Processing claim 31/188...
Processing claim 32/188...
Processing claim 33/188...
Processing claim 34/188...
Processing claim 35/188...
Processing claim 36/188...
Processing claim 37/1

In [ ]:
import json
import time
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


# --------------------------------------------------
# 1. CLEAR QWEN 3B FROM GPU
# --------------------------------------------------

try:
    del model
    del tokenizer
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print("Qwen 3B cleared from GPU.")


# --------------------------------------------------
# 2. LOAD FINAL 188-CLAIM DATA
# --------------------------------------------------

with open("final_retrieval_results.json", "r", encoding="utf-8") as f:
    final_data = json.load(f)

print(f"Loaded {len(final_data)} claims.")


# --------------------------------------------------
# 3. LOAD QWEN 2.5 7B IN 4-BIT
# --------------------------------------------------

model_name = "Qwen/Qwen2.5-7B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading Qwen 2.5 7B...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Qwen 7B loaded.")


# --------------------------------------------------
# 4. VERIFICATION FUNCTION
# --------------------------------------------------

def verify_claim(claim, documents):

    evidence_text = ""

    for i, doc in enumerate(documents, start=1):
        evidence_text += (
            f"\nDOCUMENT {i}\n"
            f"Title: {doc['title']}\n"
            f"Abstract: {doc['text']}\n"
        )

    prompt = f"""
You are verifying a scientific claim using retrieved scientific abstracts.

CLAIM:
{claim}

RETRIEVED EVIDENCE:
{evidence_text}

Based only on the retrieved evidence, classify the claim as exactly one of:

SUPPORT
CONTRADICT

Return only the label.
"""

    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated_tokens[0],
        skip_special_tokens=True
    ).strip().upper()

    if "CONTRADICT" in prediction:
        return "CONTRADICT"

    if "SUPPORT" in prediction:
        return "SUPPORT"

    return "INVALID"


# --------------------------------------------------
# 5. RUN ALL 188 CLAIMS
# --------------------------------------------------

results = []
start_time = time.time()

for number, item in enumerate(final_data, start=1):

    print(f"Processing claim {number}/{len(final_data)}...")

    bm25_prediction = verify_claim(
        item["claim"],
        item["bm25_top5"]
    )

    dense_prediction = verify_claim(
        item["claim"],
        item["dense_top5"]
    )

    results.append({
        "claim_id": item["claim_id"],
        "claim": item["claim"],
        "gold_label": item["gold_label"],
        "bm25_prediction": bm25_prediction,
        "dense_prediction": dense_prediction
    })


# --------------------------------------------------
# 6. CALCULATE ACCURACY
# --------------------------------------------------

bm25_correct = sum(
    r["bm25_prediction"] == r["gold_label"]
    for r in results
)

dense_correct = sum(
    r["dense_prediction"] == r["gold_label"]
    for r in results
)

elapsed = time.time() - start_time


print()
print("--------------------------------")
print("QWEN 7B FINAL RESULTS")
print("--------------------------------")

print(
    f"BM25 + Qwen 7B: "
    f"{bm25_correct}/188 = {bm25_correct / 188 * 100:.2f}%"
)

print(
    f"Dense + Qwen 7B: "
    f"{dense_correct}/188 = {dense_correct / 188 * 100:.2f}%"
)

print(f"Total generation time: {elapsed:.2f} seconds")


# --------------------------------------------------
# 7. SAVE RESULTS
# --------------------------------------------------

with open("qwen_7b_final_results.json", "w", encoding="utf-8") as f:
    json.dump(
        results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved to qwen_7b_final_results.json")

Qwen 3B cleared from GPU.
Loaded 188 claims.
Loading Qwen 2.5 7B...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen 7B loaded.
Processing claim 1/188...
Processing claim 2/188...
Processing claim 3/188...
Processing claim 4/188...
Processing claim 5/188...
Processing claim 6/188...
Processing claim 7/188...
Processing claim 8/188...
Processing claim 9/188...
Processing claim 10/188...
Processing claim 11/188...
Processing claim 12/188...
Processing claim 13/188...
Processing claim 14/188...
Processing claim 15/188...
Processing claim 16/188...
Processing claim 17/188...
Processing claim 18/188...
Processing claim 19/188...
Processing claim 20/188...
Processing claim 21/188...
Processing claim 22/188...
Processing claim 23/188...
Processing claim 24/188...
Processing claim 25/188...
Processing claim 26/188...
Processing claim 27/188...
Processing claim 28/188...
Processing claim 29/188...
Processing claim 30/188...
Processing claim 31/188...
Processing claim 32/188...
Processing claim 33/188...
Processing claim 34/188...
Processing claim 35/188...
Processing claim 36/188...
Processing claim 37/1

In [ ]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.1 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Loaded:", model.config._name_or_path)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-3B-Instruct


In [ ]:
import json
import time
import torch

# Load Hybrid RRF retrieval results
with open("hybrid_rrf_results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

correct = 0
results = []

start_time = time.time()

for i, item in enumerate(data, start=1):

    claim = item["claim"]
    gold_label = item["gold_label"]

    evidence = "\n\n".join(
        f"Title: {doc['title']}\nAbstract: {doc['text']}"
        for doc in item["hybrid_top5"]
    )

    prompt = f"""You are verifying a scientific claim using retrieved scientific abstracts.

CLAIM:
{claim}

RETRIEVED EVIDENCE:
{evidence}

Based only on the retrieved evidence, classify exactly one of:

SUPPORT
CONTRADICT

Return only the label."""

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip().upper()

    if "CONTRADICT" in prediction:
        prediction = "CONTRADICT"
    elif "SUPPORT" in prediction:
        prediction = "SUPPORT"

    if prediction == gold_label:
        correct += 1

    results.append({
        "claim_id": item["claim_id"],
        "claim": claim,
        "gold_label": gold_label,
        "prediction": prediction
    })

    print(f"Processing claim {i}/{len(data)}...")


elapsed = time.time() - start_time
accuracy = correct / len(data)

print()
print("--------------------------------")
print("RRF + QWEN 2.5 3B")
print("--------------------------------")
print(f"Correct: {correct}/{len(data)}")
print(f"Accuracy: {accuracy:.2%}")
print(f"Generation time: {elapsed:.2f} seconds")


with open("rrf_qwen_3b_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to: rrf_qwen_3b_results.json")

Processing claim 1/188...
Processing claim 2/188...
Processing claim 3/188...
Processing claim 4/188...
Processing claim 5/188...
Processing claim 6/188...
Processing claim 7/188...
Processing claim 8/188...
Processing claim 9/188...
Processing claim 10/188...
Processing claim 11/188...
Processing claim 12/188...
Processing claim 13/188...
Processing claim 14/188...
Processing claim 15/188...
Processing claim 16/188...
Processing claim 17/188...
Processing claim 18/188...
Processing claim 19/188...
Processing claim 20/188...
Processing claim 21/188...
Processing claim 22/188...
Processing claim 23/188...
Processing claim 24/188...
Processing claim 25/188...
Processing claim 26/188...
Processing claim 27/188...
Processing claim 28/188...
Processing claim 29/188...
Processing claim 30/188...
Processing claim 31/188...
Processing claim 32/188...
Processing claim 33/188...
Processing claim 34/188...
Processing claim 35/188...
Processing claim 36/188...
Processing claim 37/188...
Processing

In [ ]:
del model
del tokenizer

import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("3B cleared.")

3B cleared.


In [9]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Clear previous model from GPU if it exists
try:
    del model
    del tokenizer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

model_name = "Qwen/Qwen2.5-7B-Instruct"

# 4-bit loading for memory efficiency
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Loading: {model_name} with 4-bit quantization...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

print("Model loaded successfully!")
print("Device:", model.device)
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Loading: Qwen/Qwen2.5-7B-Instruct with 4-bit quantization...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully!
Device: cuda:0
GPU memory allocated: 5.46 GB


In [10]:
import json
import time
import torch

with open("hybrid_rrf_results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

correct = 0
results = []

start_time = time.time()

for i, item in enumerate(data, start=1):

    claim = item["claim"]
    gold_label = item["gold_label"]

    evidence = "\n\n".join(
        f"Title: {doc['title']}\nAbstract: {doc['text']}"
        for doc in item["hybrid_top5"]
    )

    prompt = f"""You are verifying a scientific claim using retrieved scientific abstracts.

CLAIM:
{claim}

RETRIEVED EVIDENCE:
{evidence}

Based only on the retrieved evidence, classify exactly one of:

SUPPORT
CONTRADICT

Return only the label."""

    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip().upper()

    if "CONTRADICT" in prediction:
        prediction = "CONTRADICT"
    elif "SUPPORT" in prediction:
        prediction = "SUPPORT"

    if prediction == gold_label:
        correct += 1

    results.append({
        "claim_id": item["claim_id"],
        "claim": claim,
        "gold_label": gold_label,
        "prediction": prediction
    })

    print(f"Processing claim {i}/{len(data)}...")

elapsed = time.time() - start_time
accuracy = correct / len(data)

print()
print("--------------------------------")
print("RRF + QWEN 2.5 7B")
print("--------------------------------")
print(f"Correct: {correct}/{len(data)}")
print(f"Accuracy: {accuracy:.2%}")
print(f"Generation time: {elapsed:.2f} seconds")

with open("rrf_qwen_7b_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to: rrf_qwen_7b_results.json")

Processing claim 1/188...
Processing claim 2/188...
Processing claim 3/188...
Processing claim 4/188...
Processing claim 5/188...
Processing claim 6/188...
Processing claim 7/188...
Processing claim 8/188...
Processing claim 9/188...
Processing claim 10/188...
Processing claim 11/188...
Processing claim 12/188...
Processing claim 13/188...
Processing claim 14/188...
Processing claim 15/188...
Processing claim 16/188...
Processing claim 17/188...
Processing claim 18/188...
Processing claim 19/188...
Processing claim 20/188...
Processing claim 21/188...
Processing claim 22/188...
Processing claim 23/188...
Processing claim 24/188...
Processing claim 25/188...
Processing claim 26/188...
Processing claim 27/188...
Processing claim 28/188...
Processing claim 29/188...
Processing claim 30/188...
Processing claim 31/188...
Processing claim 32/188...
Processing claim 33/188...
Processing claim 34/188...
Processing claim 35/188...
Processing claim 36/188...
Processing claim 37/188...
Processing